# HR — Is the Company Paying People Fairly? (DS_Day01_11)
**Objective:** compensation-pattern analysis (not salary prediction). Linear Regression is used only as a supporting model.

**Dataset:** `DS_Day01_11_HR_Salary_Inconsistent_Dataset.csv` — GitHub repo `suryap-0812/DS-Hackathon` (folder `Datasets/`).
The repo README describes it as a hackathon dataset *intentionally built with inconsistencies* (i.e. synthetic/practice data),
so findings describe **this dataset**, not a real company. Salary currency is not stated; values are annual and the locations are
Indian cities, so INR is assumed. `Salary_Increment` is a percentage.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns, joblib
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RS = 42
RAW, CLEAN = "../data/raw_dataset.csv", "../data/dataset.csv"
OUT, VIZ, MODEL = "../outputs/", "../outputs/visualizations/", "../model/linear_regression_model.pkl"
pd.set_option("display.width", 200, "display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

## Task 1 — Data cleaning & validation

In [2]:
raw = pd.read_csv(RAW)
print("Raw shape:", raw.shape)
print("\nDtypes:\n", raw.dtypes.to_string())
print("\nMissing values:\n", raw.isna().sum()[raw.isna().sum() > 0].to_string())
print("\nExact duplicate rows:", raw.duplicated().sum(), "| duplicated Employee_IDs:", raw.Employee_ID.duplicated().sum())
print("Employee_ID format invalid:", (~raw.Employee_ID.astype(str).str.match(r"^EMP_\d{3}$")).sum())
print("Experience < 0:", (raw.Experience < 0).sum(), "| Experience > 40:", (raw.Experience > 40).sum())
print("Salary <= 0:", (raw.Salary <= 0).sum())
print("Performance_Score outside 1-5:", (~raw.Performance_Score.between(1, 5) & raw.Performance_Score.notna()).sum())
print("Salary_Increment outside 0-15%:", (~raw.Salary_Increment.between(0, 15)).sum())
for c in ["Designation", "Education", "Location"]:
    print(f"\n{c} raw values:", sorted(raw[c].dropna().unique().tolist(), key=str))

Raw shape: (270, 8)

Dtypes:
 Employee_ID              str
Designation              str
Experience             int64
Education                str
Location                 str
Performance_Score    float64
Salary               float64
Salary_Increment     float64

Missing values:
 Designation          5
Education            4
Location             4
Performance_Score    5
Salary               5

Exact duplicate rows: 10 | duplicated Employee_IDs: 10
Employee_ID format invalid: 0
Experience < 0: 2 | Experience > 40: 1
Salary <= 0: 2
Performance_Score outside 1-5: 2
Salary_Increment outside 0-15%: 2

Designation raw values: [' Senior Software Engineer ', 'Data Analyst', 'HR Executive', 'Intern', 'JUNIOR DEVELOPER', 'Junior Developer', 'Project Manager', 'Senior Software Engineer', 'Software Engineer', 'Tech Lead', 'Unknown', 'software engineer']

Education raw values: ["Bachelor's", 'Diploma', 'MBA', 'MBA ', "Master's", 'PhD', 'Unknown', 'bachelors', "master's "]

Location raw values: [' ba

In [3]:
log = []
df = raw.copy()

# 1) categorical consistency: strip/case/aliases; 'Unknown' is not a real value -> missing
DES = {"junior developer": "Junior Developer", "software engineer": "Software Engineer",
       "senior software engineer": "Senior Software Engineer", "hr executive": "HR Executive",
       "tech lead": "Tech Lead", "project manager": "Project Manager", "data analyst": "Data Analyst", "intern": "Intern"}
EDU = {"bachelor's": "Bachelor's", "bachelors": "Bachelor's", "master's": "Master's", "masters": "Master's",
       "mba": "MBA", "phd": "PhD", "diploma": "Diploma"}
LOC = {k: k.title() for k in ["bangalore", "hyderabad", "mumbai", "chennai", "pune", "delhi", "remote"]}
changed = 0
for col, mp in [("Designation", DES), ("Education", EDU), ("Location", LOC)]:
    key = df[col].astype("string").str.strip().str.lower()
    changed += int((df[col].notna() & (df[col] != key.map(mp))).sum())
    df[col] = key.map(mp)   # anything not in mapping (e.g. 'unknown') becomes NaN
log.append(("Categorical values standardised (case/whitespace/aliases) or 'Unknown' -> missing", changed, "cell values"))

# 2) exact duplicates
n = df.duplicated().sum(); df = df.drop_duplicates().reset_index(drop=True)
log.append(("Exact duplicate rows removed", int(n), "rows removed"))
assert df.Employee_ID.is_unique

# 3) invalid numeric values -> set to missing (never imputed)
inv_perf = ~df.Performance_Score.between(1, 5) & df.Performance_Score.notna()
inv_inc = ~df.Salary_Increment.between(0, 15)
df.loc[inv_perf, "Performance_Score"] = np.nan
df.loc[inv_inc, "Salary_Increment"] = np.nan
log.append(("Performance_Score outside 1-5 set to missing (row kept)", int(inv_perf.sum()), "values"))
log.append(("Salary_Increment outside 0-15% (-3, 25) set to missing (row kept)", int(inv_inc.sum()), "values"))

# 4) rows lacking a valid core field cannot be analysed for pay patterns -> removed (reported, not imputed)
core_bad = (df.Experience < 0) | (df.Experience > 40) | (df.Salary <= 0) | df[["Designation", "Education", "Location", "Salary"]].isna().any(axis=1)
reasons = {"Experience invalid (<0 or >40)": ((df.Experience < 0) | (df.Experience > 40)).sum(),
           "Salary missing or <=0": (df.Salary.isna() | (df.Salary <= 0)).sum(),
           "Designation missing/Unknown": df.Designation.isna().sum(),
           "Education missing/Unknown": df.Education.isna().sum(),
           "Location missing/Unknown": df.Location.isna().sum()}
for k, v in reasons.items(): log.append((f"  reason (may overlap): {k}", int(v), "rows"))
log.append(("Rows removed for invalid/missing core field", int(core_bad.sum()), "rows removed"))
df = df[~core_bad].reset_index(drop=True)
log.append(("Rows retained (final cleaned dataset)", len(df), f"of {len(raw)} raw ({len(df)/len(raw):.1%})"))
print(pd.DataFrame(log, columns=["Step", "Count", "Unit"]).to_string(index=False))

print("\nRemaining missing:", df.isna().sum()[df.isna().sum() > 0].to_dict(), "(Performance_Score/Increment kept as missing; excluded only where needed)")
# basic outlier inspection (IQR fences) - flagged, retained if plausible
for c in ["Salary", "Experience", "Salary_Increment"]:
    q1, q3 = df[c].quantile([.25, .75]); f = 1.5 * (q3 - q1)
    print(f"IQR outliers in {c}: {((df[c] < q1 - f) | (df[c] > q3 + f)).sum()}")
print("Note: Junior Developer with unusually high experience retained (valid value, examined in peer analysis):")
print(df[(df.Designation == "Junior Developer") & (df.Experience > 8)][["Employee_ID", "Experience", "Salary"]].to_string(index=False))

pd.DataFrame(log, columns=["Step", "Count", "Unit"]).to_csv(OUT + "cleaning_log.csv", index=False)
df.to_csv(CLEAN, index=False)
print("\nSaved cleaned dataset:", df.shape)

                                                                             Step  Count               Unit
Categorical values standardised (case/whitespace/aliases) or 'Unknown' -> missing     15        cell values
                                                     Exact duplicate rows removed     10       rows removed
                          Performance_Score outside 1-5 set to missing (row kept)      2             values
                Salary_Increment outside 0-15% (-3, 25) set to missing (row kept)      2             values
                             reason (may overlap): Experience invalid (<0 or >40)      3               rows
                                      reason (may overlap): Salary missing or <=0      7               rows
                                reason (may overlap): Designation missing/Unknown      6               rows
                                  reason (may overlap): Education missing/Unknown      5               rows
                            

## Task 2 — Descriptive statistics

In [4]:
bands = [-1, 1, 3, 6, 10, 100]; blabels = ["0-1", "2-3", "4-6", "7-10", "11+"]
df["Exp_Band"] = pd.cut(df.Experience, bins=bands, labels=blabels)
num = ["Salary", "Experience", "Salary_Increment", "Performance_Score"]
desc = df[num].agg(["count", "mean", "median", "min", "max", "std"]).T.round(2)
print(desc.to_string()); desc.to_csv(OUT + "descriptive_overall.csv")
for c in ["Designation", "Education", "Location"]:
    print(f"\n{c} counts:", df[c].value_counts().to_dict())

def grp(by):
    g = df.groupby(by, observed=True).agg(n=("Salary", "size"), mean_salary=("Salary", "mean"), median_salary=("Salary", "median"),
        min_salary=("Salary", "min"), max_salary=("Salary", "max"), std_salary=("Salary", "std"),
        avg_experience=("Experience", "mean"), avg_increment=("Salary_Increment", "mean"))
    return g.round(1)
for by in ["Education", "Exp_Band"]:
    t = grp(by); print(f"\nSalary stats by {by}:\n", t.to_string()); t.to_csv(OUT + f"stats_by_{by.lower()}.csv")

                   count        mean    median       min        max        std
Salary             234.0  1218273.50  969500.0  284000.0  3148000.0  659288.16
Experience         234.0        4.41       3.0       0.0       17.0       4.29
Salary_Increment   233.0        6.67       6.7       0.0       12.0       2.56
Performance_Score  228.0        3.32       3.0       1.0        5.0       1.19

Designation counts: {'Software Engineer': 66, 'Junior Developer': 42, 'Senior Software Engineer': 29, 'HR Executive': 28, 'Tech Lead': 26, 'Data Analyst': 21, 'Project Manager': 21, 'Intern': 1}

Education counts: {"Bachelor's": 129, "Master's": 67, 'MBA': 30, 'PhD': 7, 'Diploma': 1}

Location counts: {'Bangalore': 58, 'Hyderabad': 50, 'Mumbai': 40, 'Chennai': 35, 'Pune': 31, 'Delhi': 19, 'Remote': 1}

Salary stats by Education:
               n  mean_salary  median_salary  min_salary  max_salary  std_salary  avg_experience  avg_increment
Education                                                  

## Task 3 — Salary differences across job roles

In [5]:
role = grp("Designation").sort_values("median_salary", ascending=False)
print(role.to_string()); role.to_csv(OUT + "salary_by_designation.csv")
# Context: raw role gaps are confounded by experience -> compare roles inside the overlapping 2-6 year range
ov = df[df.Experience.between(2, 6)]
ovt = ov.groupby("Designation").agg(n=("Salary", "size"), avg_exp=("Experience", "mean"), median_salary=("Salary", "median")).round(1)
print("\nRoles compared at overlapping experience (2-6 yrs):\n", ovt[ovt.n >= 5].sort_values("median_salary", ascending=False).to_string())
print("\nExperience range by role:\n", df.groupby("Designation").Experience.agg(["min", "median", "max"]).to_string())
ovt.to_csv(OUT + "salary_by_designation_exp_2_6.csv")

                           n  mean_salary  median_salary  min_salary  max_salary  std_salary  avg_experience  avg_increment
Designation                                                                                                                
Tech Lead                 26    2514038.5      2501000.0   2022000.0   3148000.0    280523.6            13.0            5.6
Senior Software Engineer  29    1743482.8      1734000.0    831000.0   2950000.0    371387.5             8.5            5.6
Project Manager           21    1739047.6      1715000.0    891000.0   2113000.0    268858.4             4.8            6.8
Software Engineer         66    1035393.9      1013500.0    630000.0   2005000.0    215246.5             3.8            7.1
Intern                     1     785000.0       785000.0    785000.0    785000.0         NaN             1.0           10.7
Data Analyst              21     814619.0       774000.0    564000.0   1015000.0    116856.5             0.6            7.5
HR Execu

## Task 4 — Salary growth with experience

In [6]:
r, p = stats.pearsonr(df.Experience, df.Salary); rho, prho = stats.spearmanr(df.Experience, df.Salary)
print(f"Experience vs Salary: Pearson r={r:.3f} (p={p:.2g}), Spearman rho={rho:.3f}  (n={len(df)})")
eb = grp("Exp_Band")[["n", "mean_salary", "median_salary", "min_salary", "max_salary", "std_salary"]]
print("\nSalary by experience band:\n", eb.to_string()); eb.to_csv(OUT + "salary_by_experience_band.csv")
# Within-role association (removes the role/experience overlap)
rows = []
for d, g in df.groupby("Designation"):
    if len(g) >= 20 and g.Experience.nunique() > 2:
        rr, pp = stats.spearmanr(g.Experience, g.Salary)
        slope = np.polyfit(g.Experience, g.Salary, 1)[0]
        rows.append((d, len(g), round(rr, 3), round(pp, 3), round(slope)))
within = pd.DataFrame(rows, columns=["Designation", "n", "spearman_rho", "p_value", "slope_salary_per_year"])
print("\nWithin-role association (roles with n>=20):\n", within.to_string(index=False)); within.to_csv(OUT + "experience_salary_within_role.csv", index=False)

Experience vs Salary: Pearson r=0.903 (p=2.8e-87), Spearman rho=0.879  (n=234)

Salary by experience band:
            n  mean_salary  median_salary  min_salary  max_salary  std_salary
Exp_Band                                                                    
0-1       66     658197.0       648500.0    323000.0   1015000.0    156471.2
2-3       62     916532.3       884000.0    284000.0   1852000.0    280865.3
4-6       58    1344862.1      1142500.0    630000.0   2113000.0    391171.9
7-10      22    1958272.7      1851000.0    831000.0   2950000.0    439348.8
11+       26    2451000.0      2479500.0   1756000.0   3148000.0    361237.3

Within-role association (roles with n>=20):
              Designation  n  spearman_rho  p_value  slope_salary_per_year
            Data Analyst 21         0.573    0.007                 101617
            HR Executive 28         0.650    0.000                  77927
        Junior Developer 42         0.228    0.147                 129204
         Pr

## Task 5 — Peer compensation comparison
**Peer definition (transparent):** for each employee, peers = *other* employees with the **same Designation** and Experience within **±2 years**.
If fewer than 5 such peers exist, fall back to *same Designation only* (labelled). If the Designation has <5 other employees, the employee is not assessed.
Location/Education are not in the peer key because groups would fall below 5 people; their association with salary is examined in Tasks 6 and 11.
**Rule:** |Salary difference %| ≥ 25% → "Substantial Peer Deviation / Compensation Review Candidate" (a screening threshold, not a fairness verdict).

In [7]:
MIN_PEERS, THRESH = 5, 25.0
def peers(row):
    same = df[(df.Designation == row.Designation) & (df.Employee_ID != row.Employee_ID)]
    win = same[(same.Experience - row.Experience).abs() <= 2]
    if len(win) >= MIN_PEERS: return win.Salary.median(), len(win), "Designation +/-2 yrs experience"
    if len(same) >= MIN_PEERS: return same.Salary.median(), len(same), "Designation only (fallback)"
    return np.nan, len(same), "Not assessed (too few peers)"
res = df.apply(peers, axis=1, result_type="expand"); res.columns = ["Peer_Median_Salary", "Peer_N", "Peer_Basis"]
pe = pd.concat([df, res], axis=1)
pe["Salary_Diff"] = pe.Salary - pe.Peer_Median_Salary
pe["Salary_Diff_Pct"] = (pe.Salary_Diff / pe.Peer_Median_Salary * 100).round(1)
pe["Review_Flag"] = np.where(pe.Salary_Diff_Pct >= THRESH, "Review Candidate - above peer median",
                     np.where(pe.Salary_Diff_Pct <= -THRESH, "Review Candidate - below peer median", "Within range"))
pe.loc[pe.Peer_Median_Salary.isna(), "Review_Flag"] = "Not assessed"
print("Peer basis:", pe.Peer_Basis.value_counts().to_dict())
print("\nSalary_Diff_Pct distribution (assessed employees):\n", pe.Salary_Diff_Pct.describe().round(1).to_string())
print("\nReview flags:", pe.Review_Flag.value_counts().to_dict())
for t in (15, 20, 25, 30, 40):   # sensitivity of the threshold
    print(f"  threshold ±{t}%: {(pe.Salary_Diff_Pct.abs() >= t).sum()} employees")
cols = ["Employee_ID", "Designation", "Experience", "Education", "Location", "Performance_Score", "Salary", "Peer_Median_Salary", "Peer_N", "Peer_Basis", "Salary_Diff", "Salary_Diff_Pct", "Review_Flag"]
cand = pe[pe.Review_Flag.str.startswith("Review")].sort_values("Salary_Diff_Pct", key=abs, ascending=False)[cols]
print("\nTop 12 largest deviations:\n", cand.head(12).to_string(index=False))
print("\nCandidates by designation:\n", pd.crosstab(cand.Designation, cand.Review_Flag).to_string())
ass = pe[pe.Review_Flag != "Not assessed"]
rate = ass.groupby("Designation").agg(n=("Salary", "size"), median_salary=("Salary", "median"),
    flagged=("Review_Flag", lambda x: int(x.str.startswith("Review").sum())),
    iqr_of_diff_pct=("Salary_Diff_Pct", lambda x: x.quantile(.75) - x.quantile(.25)))
rate["flag_rate_%"] = (rate.flagged / rate.n * 100).round(1); rate["iqr_of_diff_pct"] = rate.iqr_of_diff_pct.round(1)
print("\nFlag rate and within-role dispersion of peer deviation by designation:\n", rate.to_string()); rate.to_csv(OUT + "peer_flag_rate_by_designation.csv")
print("Fallback-basis candidates:", (cand.Peer_Basis != "Designation +/-2 yrs experience").sum())
pe[cols].to_csv(OUT + "peer_comparison_all_employees.csv", index=False)
cand.to_csv(OUT + "compensation_review_candidates.csv", index=False)

Peer basis: {'Designation +/-2 yrs experience': 232, 'Not assessed (too few peers)': 1, 'Designation only (fallback)': 1}

Salary_Diff_Pct distribution (assessed employees):
 count    233.0
mean       2.8
std       30.8
min      -53.7
25%       -8.4
50%       -1.1
75%        9.9
max      355.8

Review flags: {'Within range': 190, 'Review Candidate - above peer median': 24, 'Review Candidate - below peer median': 19, 'Not assessed': 1}
  threshold ±15%: 76 employees
  threshold ±20%: 56 employees
  threshold ±25%: 43 employees
  threshold ±30%: 30 employees
  threshold ±40%: 13 employees

Top 12 largest deviations:
 Employee_ID              Designation  Experience  Education  Location  Performance_Score    Salary  Peer_Median_Salary  Peer_N                      Peer_Basis  Salary_Diff  Salary_Diff_Pct                          Review_Flag
    EMP_213         Junior Developer          16   Master's Bangalore                NaN 2744000.0            602000.0      41     Designation only (fa

## Task 6 — Salary structures across locations

In [8]:
loc = grp("Location").sort_values("median_salary", ascending=False)
senior = ["Tech Lead", "Project Manager", "Senior Software Engineer"]
loc["senior_role_share_%"] = (df.assign(s=df.Designation.isin(senior)).groupby("Location").s.mean() * 100).round(1)
loc["median_peer_dev_%"] = pe.groupby("Location").Salary_Diff_Pct.median().round(1)
loc["review_candidates"] = cand.groupby("Location").size().reindex(loc.index).fillna(0).astype(int)
print(loc.to_string()); loc.to_csv(OUT + "salary_by_location.csv")
print("\nReview-candidate direction by location:\n", pd.crosstab(cand.Location, cand.Review_Flag.str.replace("Review Candidate - ", "")).to_string())
print("\nRaw median-salary spread across major locations (n>=20):", end=" ")
big = loc[loc.n >= 20]; print(f"{big.median_salary.min():,.0f} to {big.median_salary.max():,.0f}")
print("Median salary by Location x Designation (n>=3 cells):")
lt = df.groupby(["Designation", "Location"]).Salary.agg(["size", "median"]); lt = lt[lt["size"] >= 3]["median"].unstack().round(0)
print(lt.to_string())
kw = stats.kruskal(*[g.Salary.values for _, g in df[df.Location.isin(big.index)].groupby("Location")])
print(f"\nKruskal-Wallis on raw salary across major locations: H={kw.statistic:.2f}, p={kw.pvalue:.3f}")
kw2 = stats.kruskal(*[g.Salary_Diff_Pct.dropna().values for _, g in pe[pe.Location.isin(big.index)].groupby("Location")])
print(f"Kruskal-Wallis on peer-adjusted deviation % across major locations: H={kw2.statistic:.2f}, p={kw2.pvalue:.3f}")

            n  mean_salary  median_salary  min_salary  max_salary  std_salary  avg_experience  avg_increment  senior_role_share_%  median_peer_dev_%  review_candidates
Location                                                                                                                                                               
Delhi      19    1320000.0      1110000.0    414000.0   2325000.0    605319.0             5.4            5.9                 47.4               -3.2                  2
Bangalore  58    1402517.2      1108000.0    470000.0   3148000.0    734973.4             5.1            6.8                 39.7                9.8                 13
Mumbai     40    1336125.0      1090000.0    404000.0   2773000.0    678359.2             4.5            5.9                 35.0                6.6                  7
Hyderabad  50    1053620.0       929000.0    419000.0   2865000.0    596468.5             3.5            7.0                 24.0               -2.1            

## Task 7 — Performance score vs salary increment

In [9]:
pf = df.dropna(subset=["Performance_Score", "Salary_Increment"])
r7, p7 = stats.pearsonr(pf.Performance_Score, pf.Salary_Increment); rho7, prho7 = stats.spearmanr(pf.Performance_Score, pf.Salary_Increment)
print(f"Performance vs Increment (n={len(pf)}): Pearson r={r7:.3f} (p={p7:.2g}), Spearman rho={rho7:.3f} (p={prho7:.2g})")
pt = pf.groupby("Performance_Score").Salary_Increment.agg(n="size", mean="mean", median="median", std="std", min="min", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75), max="max").round(2)
print(pt.to_string()); pt.to_csv(OUT + "increment_by_performance.csv")
print("Performance vs Salary (for context): Pearson r=%.3f" % stats.pearsonr(pf.Performance_Score, pf.Salary)[0])

Performance vs Increment (n=227): Pearson r=0.845 (p=3.6e-63), Spearman rho=0.844 (p=1.1e-62)
                    n  mean  median   std  min   q25    q75   max
Performance_Score                                                
1.0                21  2.05     2.0  1.32  0.0  1.30   3.30   4.1
2.0                32  4.51     4.6  1.24  1.0  4.07   5.12   7.4
3.0                67  6.25     6.4  1.44  3.1  5.45   7.10  10.0
4.0                66  7.69     7.8  1.47  1.0  6.93   8.60  10.3
5.0                41  9.85     9.9  1.25  7.8  8.60  10.70  12.0
Performance vs Salary (for context): Pearson r=-0.090


## Task 8 — Four main visualizations

In [10]:
d = df.assign(Salary_L=df.Salary / 1e5)   # lakhs, plotting only
order = d.groupby("Designation").Salary_L.median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=d, x="Designation", y="Salary_L", order=order, palette="Blues", showfliers=False, ax=ax)
sns.stripplot(data=d, x="Designation", y="Salary_L", order=order, color="k", size=2.5, alpha=.5, ax=ax)
ax.set_xticklabels([f"{t}\n(n={int((d.Designation == t).sum())})" for t in order], rotation=0, fontsize=8)
ax.set_title("Viz 1: Salary Distribution by Designation"); ax.set_ylabel("Annual salary (lakhs; INR assumed)"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(VIZ + "1_salary_by_designation.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=d, x="Experience", y="Salary_L", hue="Designation", hue_order=order, palette="tab10", s=35, alpha=.8, ax=ax)
sns.regplot(data=d, x="Experience", y="Salary_L", scatter=False, color="black", line_kws={"lw": 2, "ls": "--"}, ax=ax)
ax.set_title(f"Viz 2: Experience vs Salary (overall linear trend, Pearson r = {r:.2f})")
ax.set_xlabel("Experience (years)"); ax.set_ylabel("Annual salary (lakhs; INR assumed)"); ax.legend(fontsize=8, title="Designation")
plt.tight_layout(); plt.savefig(VIZ + "2_experience_vs_salary.png", dpi=150); plt.close()

lorder = d.groupby("Location").Salary_L.median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=d, x="Location", y="Salary_L", order=lorder, palette="Greens", showfliers=False, ax=ax)
sns.stripplot(data=d, x="Location", y="Salary_L", order=lorder, color="k", size=2.5, alpha=.5, ax=ax)
ax.set_xticklabels([f"{t}\n(n={int((d.Location == t).sum())})" for t in lorder], fontsize=9)
ax.set_title("Viz 3: Salary Comparison Across Locations (not adjusted for role/experience mix)"); ax.set_ylabel("Annual salary (lakhs; INR assumed)"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(VIZ + "3_salary_by_location.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=pf, x="Performance_Score", y="Salary_Increment", palette="Oranges", showfliers=False, ax=ax)
sns.stripplot(data=pf, x="Performance_Score", y="Salary_Increment", color="k", size=2.5, alpha=.5, ax=ax)
ax.set_title(f"Viz 4: Performance Score vs Salary Increment (Pearson r = {r7:.2f}, n = {len(pf)})")
ax.set_xlabel("Performance score"); ax.set_ylabel("Salary increment (%)")
plt.tight_layout(); plt.savefig(VIZ + "4_performance_vs_increment.png", dpi=150); plt.close()
print("Saved 4 visualizations")

Saved 4 visualizations


## Tasks 9–10 — Linear Regression (supporting analytical model) and evaluation

In [11]:
feat_cat = ["Designation", "Education", "Location"]; feat_num = ["Experience", "Performance_Score"]
md = df.dropna(subset=["Performance_Score"]).copy()
print(f"Modelling rows: {len(md)} (rows with missing Performance_Score excluded, not imputed)")
X, y = md[feat_cat + feat_num], md.Salary
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RS)
# reference categories = most common, so coefficients read as "difference vs reference"
ref = [["Software Engineer"], ["Bachelor's"], ["Bangalore"]]
ohe = OneHotEncoder(drop=[r[0] for r in ref], handle_unknown="ignore", sparse_output=False)
pre = ColumnTransformer([("cat", ohe, feat_cat), ("num", "passthrough", feat_num)])
model = Pipeline([("prep", pre), ("lr", LinearRegression())]).fit(Xtr, ytr)
pred = model.predict(Xte)
mae, rmse, r2 = mean_absolute_error(yte, pred), np.sqrt(mean_squared_error(yte, pred)), r2_score(yte, pred)
cv = cross_val_score(Pipeline([("prep", pre), ("lr", LinearRegression())]), X, y, cv=5, scoring="r2")
print(f"Train rows={len(Xtr)}, Test rows={len(Xte)}, random_state={RS}")
print(f"TEST  MAE = {mae:,.0f} | RMSE = {rmse:,.0f} | R2 = {r2:.3f}")
print(f"Train R2 = {model.score(Xtr, ytr):.3f} | 5-fold CV R2 = {cv.mean():.3f} (±{cv.std():.3f})  [stability check, same pipeline]")
print(f"Context: salary mean={y.mean():,.0f}, std={y.std():,.0f}; MAE is {mae/y.mean():.1%} of mean salary")
# Management hypothesis check: same algorithm, restricted features (Designation + Experience only)
pre2 = ColumnTransformer([("cat", OneHotEncoder(drop=["Software Engineer"], handle_unknown="ignore", sparse_output=False), ["Designation"]), ("num", "passthrough", ["Experience"])])
m2 = Pipeline([("prep", pre2), ("lr", LinearRegression())]).fit(Xtr, ytr); p2 = m2.predict(Xte)
cv2 = cross_val_score(Pipeline([("prep", pre2), ("lr", LinearRegression())]), X, y, cv=5, scoring="r2")
print(f"Designation+Experience only: TEST MAE={mean_absolute_error(yte, p2):,.0f} RMSE={np.sqrt(mean_squared_error(yte, p2)):,.0f} R2={r2_score(yte, p2):.3f} | 5-fold CV R2={cv2.mean():.3f}")
print(f"Adding Education+Location+Performance: CV R2 gain = {cv.mean() - cv2.mean():+.3f}")
big_err = pd.Series(np.abs(yte - pred)).sort_values(ascending=False).head(3)
print("Largest test errors (abs):", [f"{md.loc[i, 'Employee_ID']} ({md.loc[i, 'Designation']}, exp {md.loc[i, 'Experience']}): {v:,.0f}" for i, v in big_err.items()])
joblib.dump(model, MODEL); print("Saved model:", MODEL)

Modelling rows: 228 (rows with missing Performance_Score excluded, not imputed)
Train rows=182, Test rows=46, random_state=42
TEST  MAE = 138,596 | RMSE = 268,817 | R2 = 0.801
Train R2 = 0.959 | 5-fold CV R2 = 0.911 (±0.043)  [stability check, same pipeline]
Context: salary mean=1,207,026, std=652,989; MAE is 11.5% of mean salary
Designation+Experience only: TEST MAE=153,872 RMSE=260,016 R2=0.814 | 5-fold CV R2=0.889
Adding Education+Location+Performance: CV R2 gain = +0.022
Largest test errors (abs): ['EMP_066 (Senior Software Engineer, exp 10): 1,162,588', 'EMP_025 (Senior Software Engineer, exp 9): 983,821', 'EMP_255 (Software Engineer, exp 4): 534,850']
Saved model: ../model/linear_regression_model.pkl


## Task 11 — Model interpretation (association only, not causal)

In [12]:
names = model.named_steps["prep"].get_feature_names_out()
coef = pd.DataFrame({"feature": [n.split("__")[1] for n in names], "coefficient": model.named_steps["lr"].coef_.round(0)})
counts = {f"{c}_{v}": n for c in feat_cat for v, n in md[c].value_counts().items()}
coef["n_in_data"] = coef.feature.map(counts)
print("Intercept:", round(model.named_steps["lr"].intercept_))
print("References: Designation=Software Engineer, Education=Bachelor's, Location=Bangalore\n")
print(coef.to_string(index=False)); coef.to_csv(OUT + "linear_regression_coefficients.csv", index=False)
pd.DataFrame({"metric": ["MAE", "RMSE", "R2_test", "R2_5fold_cv_mean", "R2_test_designation_experience_only", "R2_cv_designation_experience_only"], "value": [round(mae), round(rmse), round(r2, 4), round(cv.mean(), 4), round(r2_score(yte, p2), 4), round(cv2.mean(), 4)]}).to_csv(OUT + "model_evaluation.csv", index=False)

Intercept: 652513
References: Designation=Software Engineer, Education=Bachelor's, Location=Bangalore

                             feature  coefficient  n_in_data
            Designation_Data Analyst     -39287.0       21.0
            Designation_HR Executive    -227008.0       28.0
                  Designation_Intern    -185994.0        1.0
        Designation_Junior Developer    -242092.0       41.0
         Designation_Project Manager     617071.0       21.0
Designation_Senior Software Engineer     380502.0       29.0
               Designation_Tech Lead     831550.0       24.0
                   Education_Diploma      32893.0        1.0
                       Education_MBA      87532.0       30.0
                  Education_Master's     108214.0       66.0
                       Education_PhD     292784.0        7.0
                    Location_Chennai    -235075.0       34.0
                      Location_Delhi     -96611.0       18.0
                  Location_Hyderabad    -14

## Task 12 — Key insights (all figures from this notebook's outputs; salary in INR assumed, 1 lakh = 100,000)

**Insight 1 — Role pay tiers are wide, but roles sit in different experience ranges.**
Finding: Median salary runs from 6.1 lakh (Junior Developer) and 6.6 lakh (HR Executive) up to 25.0 lakh (Tech Lead).
Evidence: Tech Leads have 8–17 yrs experience vs 0–2 yrs for Data Analysts/HR Executives. In the overlapping 2–6 yr range, Project Managers (n=21, avg 4.8 yrs) have a median of 17.2 lakh vs 10.1 lakh for Software Engineers (n=66, avg 3.8 yrs).
Interpretation: Raw role gaps mix seniority with role; a ~1-year experience difference does not explain the PM–Software Engineer gap, so role structure itself is associated with pay.

**Insight 2 — Salary is strongly associated with experience overall, but much less uniformly within roles.**
Finding: Salary shows a strong positive association with experience (Pearson r = 0.90, Spearman 0.88, n = 234); band medians rise from 6.5 lakh (0–1 yrs) to 24.8 lakh (11+ yrs).
Evidence: Within-role Spearman ranges from 0.23 (Junior Developer, p = 0.147) to 0.76 (Senior Software Engineer); Software Engineers gain ~32k per extra year vs ~100k for Data Analysts.
Interpretation: The overall correlation is inflated because senior titles coincide with higher experience; experience does not progress at one common rate within each role.

**Insight 3 — 43 of 233 assessed employees (18.5%) are Substantial Peer Deviation / Compensation Review Candidates (±25% vs peer median): 24 above, 19 below.**
Finding: Deviations are concentrated in Junior Developers: 21 of 42 flagged (50%), vs 0–24% in every other role.
Evidence: Junior Developer peer-deviation IQR is 50.8 points vs 12.5–32.6 for other roles. Largest cases: EMP_213 (Junior Developer, 16 yrs, +356%, compared to all Junior Developers as no similarly experienced peers exist), EMP_214 (+98.7%), EMP_236 (+90.5%); lowest EMP_069 (−53.7%), EMP_025 (−53.6%). Count is threshold-sensitive: 76 / 56 / 43 / 30 / 13 employees at ±15 / 20 / 25 / 30 / 40%.
Interpretation: These are screening flags, not findings of under/over-payment; Junior Developer pay appears the least consistent and is the first place to look.

**Insight 4 — Location differences persist after accounting for role and experience.**
Finding: Raw median salary is ~10.9–11.1 lakh in Delhi/Bangalore/Mumbai vs ~9.1–9.3 lakh in Hyderabad/Pune/Chennai, but raw figures are affected by mix (senior-role share: Delhi 47%, Bangalore 40% vs Hyderabad 24%).
Evidence: Median peer-adjusted deviation: Bangalore +9.8%, Mumbai +6.6%, Hyderabad −2.1%, Delhi −3.2%, Pune −3.5%, Chennai −7.8% (Kruskal-Wallis p < 0.001). Review candidates above peers: Bangalore 11 vs 2 below; Hyderabad 3 above vs 8 below. Regression vs Bangalore: Chennai −2.35 lakh, Pune −1.50, Hyderabad −1.42, Delhi −0.97, Mumbai −0.17.
Interpretation: Delhi looks high in raw terms but not after peer adjustment (workforce mix). Adjusted gaps remain for Bangalore/Mumbai vs Chennai/Hyderabad; whether they reflect an intended geographic pay policy cannot be determined from this data.

**Insight 5 — Performance is closely associated with salary increments, but not with salary levels.**
Finding: Higher performance scores are associated with higher increments in the observed dataset (r = 0.85, n = 227).
Evidence: Mean increment by score 1→5: 2.05%, 4.51%, 6.25%, 7.69%, 9.85%. Performance vs salary level: r = −0.09. A few mismatches exist, e.g. EMP_239 (Tech Lead, score 4, 1.0%), EMP_003 (score 2, 7.4%), EMP_069 (score 3, 10.0%).
Interpretation: Increments follow ratings, so pay-for-performance operates through increments; current salary levels are explained by role/experience rather than performance.

**Insight 6 — Designation + Experience already explain most salary variation (supports management's view), with a modest extra contribution from other factors.**
Finding: A Linear Regression using only Designation + Experience reaches 5-fold CV R² = 0.889; adding Education, Location and Performance raises it to 0.911 (+0.022).
Evidence: Full model, held-out test (n=46): MAE 138,596; RMSE 268,817; R² 0.801 (restricted model on the same split: R² 0.814). Largest test errors are EMP_066 and EMP_025, both also peer-flagged.
Interpretation: The model is only a description of association in this dataset; it is not a statement of what anyone should earn.

**Insight 7 — Fitted-model associations (conditional on other variables; not causal).**
Finding: Each extra year of experience is associated with ~+69k salary; roles vs Software Engineer: Tech Lead +8.3 lakh, Project Manager +6.2 lakh, Senior SE +3.8 lakh, Junior Developer −2.4 lakh, HR Executive −2.3 lakh.
Evidence: Education vs Bachelor's: Master's +1.1 lakh, MBA +0.9 lakh, PhD +2.9 lakh (n = 7 only); Performance +53k per score point. Categories with n = 1 (Intern, Diploma, Remote) are not interpretable.
Interpretation: Education and performance associations are generally smaller than role associations; the PhD estimate is unreliable due to small n.

**Answer to "Is the company paying people fairly?"** In this dataset pay is largely structured by designation and experience, but 43 individual cases, Junior Developer pay dispersion and location gaps merit review. This dataset cannot establish fairness, bias or discrimination.
